In [2]:
from datasets import load_dataset
ds = load_dataset("Jiahao004/DeepTheorem")

/Users/nataliehu/Desktop/emory/nlp lab/Iterative_Reasoning/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 120754/120754 [00:01<00:00, 69704.11 examples/s]


Local copies of DeepTheorem Data

In [7]:
ds.save_to_disk("./DeepTheorem")

Saving the dataset (3/3 shards): 100%|██████████| 120754/120754 [00:04<00:00, 25651.72 examples/s]


In [8]:
ds.save_to_disk("./Data/Processed_DeepTheorem")

Saving the dataset (3/3 shards): 100%|██████████| 120754/120754 [00:04<00:00, 29589.34 examples/s]


1. Filter DeepTheorem
- Remove easy questions (only keep q with difficulties >=6)

In [14]:
from datasets import load_from_disk

dt = load_from_disk("./Data/Processed_DeepTheorem")

In [15]:
#remove easy questions
dt = dt.filter(lambda x: x["difficulty"] >=6)

Filter: 100%|██████████| 120754/120754 [00:05<00:00, 21894.25 examples/s]


In [19]:
dt_model_test = dt["train"].shuffle(seed= 42).select(range(40))

In [21]:
dt_model_test.save_to_disk("./Data/Choose_Models")

Saving the dataset (1/1 shards): 100%|██████████| 40/40 [00:00<00:00, 599.30 examples/s]


## 2. Extract proof step-numbered subset

Keep proofs whose `proof` text is explicitly structured as numbered steps. Two patterns count:
- `Step \d+` headers (case-insensitive)
- Numbered-list markers (`1. `, `2. `, ...) **anchored to the start of a line**, with at least 2 such markers present

Anchoring to line-start (instead of matching `\d+\.` anywhere) matters a lot here: an unanchored match hits 72% of the dataset (decimals, equation/citation numbers, etc.), while the line-anchored version drops to 38% and is clean on manual inspection. Proofs with multiple *separate* numbered lists (each restarting at 1) are legitimate step-structured proofs, not false positives, so we don't require global sequential numbering, just >=2 markers.

In [ ]:
import re
from datasets import load_from_disk

dt_full = load_from_disk("./Data/Processed_DeepTheorem")["train"]

step_re = re.compile(r"Step\s*\d+", re.IGNORECASE)
list_re = re.compile(r"(?m)^\s*\d+\.\s")

def is_step_numbered(example):
    proof = example["proof"]
    return bool(step_re.search(proof)) or len(list_re.findall(proof)) >= 2

dt_step_numbered = dt_full.filter(is_step_numbered)
dt_step_numbered

In [ ]:
dt_step_numbered.save_to_disk("./Data/Processed_DeepTheorem_StepNumbered")